# 🎤 MindTrack AI v5 — Voice Emotion CNN Training

**Model:** Raw Spectrogram CNN (4-layer Conv2d + BatchNorm + AdaptiveAvgPool)

**Datasets (all FREE on Kaggle):**
- **RAVDESS** — 1,440 speech files, 24 actors, 8 emotions
- **CREMA-D** — 7,442 clips, 91 actors, 6 emotions
- **TESS** — 2,800 files, 2 actresses, 7 emotions

**Combined:** ~11,682 audio samples → 8 emotion classes

**Output:** `voice_cnn.pt` (~4 MB)

In [ ]:
# Step 0: Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 1: Install dependencies
!pip install -q librosa soundfile kaggle scikit-learn matplotlib seaborn tqdm

In [ ]:
# Step 2: Setup Kaggle credentials
# Option A: Upload your kaggle.json
# from google.colab import files
# files.upload()  # Upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Option B: Set manually
import os
os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME'  # <-- CHANGE THIS
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY'  # <-- CHANGE THIS

In [ ]:
# Step 3: Download datasets from Kaggle
!kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio -p /content/data --unzip
!kaggle datasets download -d ejlok1/cremad -p /content/data/cremad --unzip
!kaggle datasets download -d ejlok1/toronto-emotional-speech-set-tess -p /content/data/tess --unzip
print("\n✅ All datasets downloaded!")

In [ ]:
# Step 4: Inspect downloaded data
import os
for root_dir in ['/content/data']:
    for d, subdirs, files in os.walk(root_dir):
        depth = d.replace(root_dir, '').count(os.sep)
        if depth < 3:
            wav_count = len([f for f in files if f.endswith('.wav')])
            if wav_count > 0:
                print(f"{d}  → {wav_count} .wav files")

In [ ]:
import os
import glob
import logging
import json
from pathlib import Path

import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# 8 emotion classes (RAVDESS standard)
EMOTION_LABELS = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
NUM_CLASSES = len(EMOTION_LABELS)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Classes: {EMOTION_LABELS}')

In [ ]:
# Step 5: Data loading functions for all 3 datasets

def load_ravdess(data_dir='/content/data'):
    """RAVDESS: filename format 03-01-{emotion}-{intensity}-{statement}-{rep}-{actor}.wav
    Emotion: 01=neutral, 02=calm, 03=happy, 04=sad, 05=angry, 06=fearful, 07=disgust, 08=surprised
    """
    files, labels = [], []
    for wav in glob.glob(os.path.join(data_dir, '**/*.wav'), recursive=True):
        basename = os.path.basename(wav)
        parts = basename.split('-')
        if len(parts) >= 3:
            try:
                emotion_code = int(parts[2]) - 1  # 0-indexed
                if 0 <= emotion_code < NUM_CLASSES:
                    files.append(wav)
                    labels.append(emotion_code)
            except ValueError:
                continue
    print(f'RAVDESS: {len(files)} files')
    return files, labels


def load_cremad(data_dir='/content/data/cremad'):
    """CREMA-D: filename format {ActorID}_{Statement}_{Emotion}_{EmotionLevel}.wav
    Emotions: ANG, DIS, FEA, HAP, NEU, SAD
    Map to RAVDESS indices: ANG→4, DIS→6, FEA→5, HAP→2, NEU→0, SAD→3
    """
    CREMAD_MAP = {'ANG': 4, 'DIS': 6, 'FEA': 5, 'HAP': 2, 'NEU': 0, 'SAD': 3}
    files, labels = [], []
    for wav in glob.glob(os.path.join(data_dir, '**/*.wav'), recursive=True):
        basename = os.path.basename(wav)
        parts = basename.replace('.wav', '').split('_')
        if len(parts) >= 3:
            emotion_code = CREMAD_MAP.get(parts[2])
            if emotion_code is not None:
                files.append(wav)
                labels.append(emotion_code)
    print(f'CREMA-D: {len(files)} files')
    return files, labels


def load_tess(data_dir='/content/data/tess'):
    """TESS: folder names contain emotion, files are like OAF_back_angry.wav
    Emotions: angry, disgust, fear, happy, neutral, ps (pleasant surprise), sad
    Map: angry→4, disgust→6, fear→5, happy→2, neutral→0, ps→7, sad→3
    """
    TESS_MAP = {'angry': 4, 'disgust': 6, 'fear': 5, 'happy': 2, 'neutral': 0, 'ps': 7, 'sad': 3}
    files, labels = [], []
    for wav in glob.glob(os.path.join(data_dir, '**/*.wav'), recursive=True):
        basename = os.path.basename(wav).lower().replace('.wav', '')
        # Emotion is the last part after the last underscore
        parts = basename.split('_')
        if len(parts) >= 1:
            emotion_word = parts[-1]
            emotion_code = TESS_MAP.get(emotion_word)
            if emotion_code is not None:
                files.append(wav)
                labels.append(emotion_code)
    print(f'TESS: {len(files)} files')
    return files, labels


# Load all datasets
r_files, r_labels = load_ravdess()
c_files, c_labels = load_cremad()
t_files, t_labels = load_tess()

all_files = r_files + c_files + t_files
all_labels = r_labels + c_labels + t_labels
print(f'\n✅ Total: {len(all_files)} audio files across {NUM_CLASSES} classes')

# Class distribution
from collections import Counter
dist = Counter(all_labels)
for i, label in enumerate(EMOTION_LABELS):
    print(f'  {label}: {dist.get(i, 0)}')

In [ ]:
# Step 6: Dataset class — Mel Spectrogram extraction

class VoiceEmotionDataset(Dataset):
    def __init__(self, file_paths, labels, sr=16000, n_mels=128, max_len=128, augment=False):
        self.file_paths = file_paths
        self.labels = labels
        self.sr = sr
        self.n_mels = n_mels
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        try:
            y, sr = librosa.load(self.file_paths[idx], sr=self.sr, mono=True)
        except Exception:
            return torch.zeros(1, self.n_mels, self.max_len), self.labels[idx]

        if len(y) < 1024:
            y = np.pad(y, (0, 1024 - len(y)))

        # Augmentation (training only)
        if self.augment:
            # Time stretch
            if np.random.random() < 0.3:
                rate = np.random.uniform(0.85, 1.15)
                y = librosa.effects.time_stretch(y, rate=rate)
            # Pitch shift
            if np.random.random() < 0.3:
                n_steps = np.random.uniform(-2, 2)
                y = librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)
            # Add noise
            if np.random.random() < 0.3:
                noise = np.random.randn(len(y)) * 0.005
                y = y + noise

        # Mel spectrogram
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=self.n_mels, fmax=8000, hop_length=512)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        # Pad/truncate to fixed length
        if mel_spec_db.shape[1] < self.max_len:
            mel_spec_db = np.pad(mel_spec_db, ((0,0), (0, self.max_len - mel_spec_db.shape[1])), mode='constant')
        else:
            mel_spec_db = mel_spec_db[:, :self.max_len]

        # Normalize
        mel_spec_db = (mel_spec_db - mel_spec_db.mean()) / (mel_spec_db.std() + 1e-6)

        tensor = torch.FloatTensor(mel_spec_db).unsqueeze(0)  # (1, n_mels, max_len)
        return tensor, self.labels[idx]

In [ ]:
# Step 7: Model architecture (matches inference code exactly)

class SpectrogramCNN(nn.Module):
    """Raw Spectrogram CNN for voice emotion classification."""
    def __init__(self, num_classes=8):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SpectrogramCNN(num_classes=NUM_CLASSES).to(DEVICE)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {params:,}')
print(model)

In [ ]:
# Step 8: Prepare data splits

train_files, test_files, train_labels, test_labels = train_test_split(
    all_files, all_labels, test_size=0.15, random_state=42, stratify=all_labels
)
train_files, val_files, train_labels, val_labels = train_test_split(
    train_files, train_labels, test_size=0.15, random_state=42, stratify=train_labels
)

print(f'Train: {len(train_files)}')
print(f'Val:   {len(val_files)}')
print(f'Test:  {len(test_files)}')

train_dataset = VoiceEmotionDataset(train_files, train_labels, augment=True)
val_dataset = VoiceEmotionDataset(val_files, val_labels, augment=False)
test_dataset = VoiceEmotionDataset(test_files, test_labels, augment=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# Step 9: Training loop

EPOCHS = 80
LR = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_f1 = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    for specs, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False):
        specs, labels = specs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(specs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
        _, pred = outputs.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
    train_loss /= len(train_loader)
    train_acc = 100.0 * correct / total

    # Validate
    model.eval()
    val_loss, all_preds, all_true = 0.0, [], []
    with torch.no_grad():
        for specs, labels in val_loader:
            specs, labels = specs.to(DEVICE), labels.to(DEVICE)
            outputs = model(specs)
            val_loss += criterion(outputs, labels).item()
            _, pred = outputs.max(1)
            all_preds.extend(pred.cpu().numpy())
            all_true.extend(labels.cpu().numpy())
    val_loss /= len(val_loader)
    val_acc = 100.0 * np.mean(np.array(all_preds) == np.array(all_true))
    val_f1 = f1_score(all_true, all_preds, average='macro') * 100

    scheduler.step()

    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss, 'val_acc': val_acc, 'val_f1': val_f1})

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), '/content/voice_cnn_best.pt')
        marker = ' ✅ BEST'
    else:
        marker = ''

    if epoch % 5 == 0 or marker:
        print(f'Epoch {epoch}/{EPOCHS} — Train: {train_loss:.4f}/{train_acc:.1f}% | '
              f'Val: {val_loss:.4f}/{val_acc:.1f}%/F1={val_f1:.1f}%{marker}')

print(f'\n🎉 Training complete! Best validation F1: {best_f1:.2f}%')

In [ ]:
# Step 10: Evaluate on test set

model.load_state_dict(torch.load('/content/voice_cnn_best.pt', weights_only=True))
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for specs, labels in tqdm(test_loader, desc='Testing'):
        specs, labels = specs.to(DEVICE), labels.to(DEVICE)
        outputs = model(specs)
        _, pred = outputs.max(1)
        all_preds.extend(pred.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

test_acc = 100.0 * np.mean(np.array(all_preds) == np.array(all_true))
test_f1 = f1_score(all_true, all_preds, average='macro') * 100
print(f'\n📊 Test Accuracy: {test_acc:.2f}%')
print(f'📊 Test F1 (macro): {test_f1:.2f}%')

print('\nClassification Report:')
report = classification_report(all_true, all_preds, target_names=EMOTION_LABELS, output_dict=True)
print(classification_report(all_true, all_preds, target_names=EMOTION_LABELS))

In [ ]:
# Step 11: Confusion matrix

cm = confusion_matrix(all_true, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Voice Emotion CNN — Test Confusion Matrix (Acc={test_acc:.1f}%, F1={test_f1:.1f}%)')
plt.tight_layout()
plt.savefig('/content/voice_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Step 12: Training curves

epochs = [h['epoch'] for h in history]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, [h['train_loss'] for h in history], label='Train Loss')
ax1.plot(epochs, [h['val_loss'] for h in history], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, [h['train_acc'] for h in history], label='Train Acc')
ax2.plot(epochs, [h['val_acc'] for h in history], label='Val Acc')
ax2.plot(epochs, [h['val_f1'] for h in history], label='Val F1', linestyle='--')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('%')
ax2.set_title('Accuracy & F1 Score')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/voice_training_curves.png', dpi=150)
plt.show()

In [ ]:
# Step 13: Save results

results = {
    'model': 'SpectrogramCNN',
    'datasets': ['RAVDESS', 'CREMA-D', 'TESS'],
    'total_samples': len(all_files),
    'num_classes': NUM_CLASSES,
    'classes': EMOTION_LABELS,
    'test_accuracy': round(test_acc, 2),
    'test_f1_macro': round(test_f1, 2),
    'best_val_f1': round(best_f1, 2),
    'epochs_trained': EPOCHS,
    'classification_report': report,
}

with open('/content/voice_results.json', 'w') as f:
    json.dump(results, f, indent=2)

with open('/content/voice_training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print('✅ Saved: voice_cnn_best.pt, voice_results.json, voice_training_history.json')

In [ ]:
# Step 14: Download the trained model
from google.colab import files

# Download the model weights
files.download('/content/voice_cnn_best.pt')
files.download('/content/voice_results.json')
files.download('/content/voice_confusion_matrix.png')
files.download('/content/voice_training_curves.png')

print('\n📥 Downloaded! Place voice_cnn_best.pt into:')
print('   mindtrack-ai/data/pretrained/voice_cnn.pt')